# 图像分类：CNN 经典应用
> 难度标签：高级 | 预计时长：15-25分钟 | 前置知识：无学习经验


> 所属模块：09_计算机视觉 | 源文件：图像分类.py | 核心功能：CNN 图像分类、torchvision 数据集、训练评估

## 概述

图像分类是计算机视觉的基础任务——给一张图片分配一个类别标签。CNN 是图像分类的标准模型，从 AlexNet 到 ResNet 再到 Vision Transformer，精度不断提升。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

## 数学原理

### 1. 卷积运算

卷积层是 CNN 的核心。给定输入特征图 $X \in \mathbb{R}^{C_{in} \times H \times W}$，卷积核 $K \in \mathbb{R}^{C_{out} \times C_{in} \times k \times k}$，输出为：

$$Y_{c,i,j} = \sum_{c'=0}^{C_{in}-1}\sum_{m=0}^{k-1}\sum_{n=0}^{k-1} K_{c,c',m,n} \cdot X_{c', i+m, j+n} + b_c$$

关键参数：
- **padding=1**：在输入边缘填充 1 像素，保持空间尺寸不变
- **kernel_size=3**：$3 \times 3$ 卷积核

输出尺寸公式：

$$H_{out} = \frac{H_{in} + 2 \times \text{padding} - \text{kernel\_size}}{\text{stride}} + 1$$

代码中 $(28 + 2 \times 1 - 3)/1 + 1 = 28$，尺寸不变。

### 2. 池化操作

MaxPool2d(2) 在 $2 \times 2$ 区域取最大值：

$$Y_{c,i,j} = \max_{0 \leq m,n < 2} X_{c, 2i+m, 2j+n}$$

空间尺寸减半：$28 \times 28 \to 14 \times 14 \to 7 \times 7$。池化提供**平移不变性**并减少参数。

### 3. 特征图的逐层变化

| 层 | 输出形状 | 参数量 |
|----|----------|--------|
| 输入 | $(1, 28, 28)$ | 0 |
| Conv2d(1→32, 3×3) | $(32, 28, 28)$ | $32 \times (1 \times 3 \times 3 + 1) = 320$ |
| MaxPool2d(2) | $(32, 14, 14)$ | 0 |
| Conv2d(32→64, 3×3) | $(64, 14, 14)$ | $64 \times (32 \times 3 \times 3 + 1) = 18{,}496$ |
| MaxPool2d(2) | $(64, 7, 7)$ | 0 |
| Flatten | $(3136,)$ | 0 |
| FC(3136→128) | $(128,)$ | $3136 \times 128 + 128 = 401{,}536$ |
| FC(128→10) | $(10,)$ | $128 \times 10 + 10 = 1{,}290$ |

### 4. ReLU 激活函数

$$\text{ReLU}(x) = \max(0, x)$$

逐元素应用，将负值置零。引入非线性，使网络能学习复杂映射。

### 5. 交叉熵损失

多分类使用交叉熵损失：

$$\mathcal{L} = -\frac{1}{n}\sum_{i=1}^{n}\sum_{c=0}^{C-1} y_{ic} \log \hat{p}_{ic}$$

其中 $\hat{p}_{ic} = \text{softmax}(z_{ic}) = \frac{e^{z_{ic}}}{\sum_j e^{z_{ij}}}$，$y_{ic}$ 是 one-hot 标签。

### 6. 代码与数学的对应

| 代码 | 数学含义 |
|------|----------|
| `Conv2d(1, 32, 3, padding=1)` | $K \in \mathbb{R}^{32 \times 1 \times 3 \times 3}$，padding 保持尺寸 |
| `MaxPool2d(2)` | $2 \times 2$ 最大池化，空间减半 |
| `Flatten()` | $(64, 7, 7) \to (3136,)$ |
| `Linear(3136, 128)` | 全连接 $W \in \mathbb{R}^{128 \times 3136}$ |
| `nn.CrossEntropyLoss()` | softmax + 交叉熵（内置 softmax） |
| `optim.Adam(lr=1e-3)` | 自适应学习率优化器 |

### 模型定义

运行 模型定义 的代码，观察算法在该环节的行为。

In [ ]:

class SimpleCNN(nn.Module):
    """简单的卷积神经网络，用于 MNIST 手写数字分类"""

    def __init__(self):
        super().__init__()
        # 特征提取：两层卷积 + 池化
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),   # (1,28,28) → (32,28,28)
            nn.ReLU(),
            nn.MaxPool2d(2),                               # → (32,14,14)
            nn.Conv2d(32, 64, kernel_size=3, padding=1),  # → (64,14,14)
# --- 继续 ---
            nn.ReLU(),
            nn.MaxPool2d(2),                               # → (64,7,7)
        )
        # 分类器：全连接层
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10),  # 10 个数字类别
# --- 继续 ---
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

### 数据加载

运行 数据加载 的代码，观察算法在该环节的行为。

In [ ]:

def get_dataloaders(batch_size=64):
    """加载 MNIST 数据集，返回训练和测试 DataLoader"""
    transform = transforms.Compose([
        transforms.ToTensor(),  # 转为张量并归一化到 [0,1]
    ])

    train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
    test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    return train_loader, test_loader

### 训练函数

执行模型训练循环，观察损失函数的收敛过程。

In [ ]:

def train(model, train_loader, criterion, optimizer, device, epoch):
    """训练一个 epoch，返回平均损失"""
    model.train()
    total_loss = 0
    num_batches = 0
# --- 循环处理 ---
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
# --- 继续 ---
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        num_batches += 1
    avg_loss = total_loss / num_batches
# --- 输出结果 ---
    print(f"  Epoch {epoch}  训练损失: {avg_loss:.4f}")
    return avg_loss

### 评估函数

在测试集上评估模型性能，关注关键指标。

In [ ]:

def evaluate(model, test_loader, device):
    """评估模型在测试集上的准确率"""
    model.eval()
    correct = 0
    total = 0
# --- 继续 ---
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            pred = output.argmax(dim=1)  # 取概率最大的类别
# --- 继续 ---
            correct += (pred == target).sum().item()
            total += target.size(0)
    accuracy = correct / total
    return accuracy

### 主流程

运行 主流程 的代码，观察算法在该环节的行为。

In [ ]:

def main():
    # 超参数
    batch_size = 64
    lr = 1e-3
    epochs = 5
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"使用设备: {device}")

    # 加载数据
    train_loader, test_loader = get_dataloaders(batch_size)
    print(f"训练集大小: {len(train_loader.dataset)}  测试集大小: {len(test_loader.dataset)}")

    # 构建模型、损失函数、优化器
    model = SimpleCNN().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # 训练
    print("\n开始训练:")
    for epoch in range(1, epochs + 1):
        train(model, train_loader, criterion, optimizer, device, epoch)

    # 评估
    accuracy = evaluate(model, test_loader, device)
    print(f"\n测试集准确率: {accuracy * 100:.2f}%")

    # 打印模型结构摘要
    total_params = sum(p.numel() for p in model.parameters())
    print(f"模型参数量: {total_params:,}")


if __name__ == "__main__":
    main()

## 关键代码解释

```python
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])
dataset = torchvision.datasets.CIFAR10(root="./data", train=True, transform=transform)
```

## 注意事项

1. 数据增强对泛化至关重要
2. 预训练模型微调通常优于从头训练
3. 图像归一化参数需要匹配预训练模型

## 总结与延伸

以上代码展示了 **图像分类** 的完整流程。

**解读要点**：
- 关注 **准确率（Accuracy）** 和 **分类报告** 中的精确率/召回率/F1
- 如果训练准确率远高于测试准确率，说明存在过拟合
- 观察 **混淆矩阵**，找出模型最容易混淆的类别

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **迁移学习**：用 ImageNet 预训练模型微调
- **Vision Transformer**：ViT 在大规模数据上超越 CNN
- **自监督预训练**：MAE、DINO 等方法
